In [1]:
import torch
from datasets import load_dataset

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    GenerationConfig,
    pipeline
)

In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
config = AutoConfig.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

In [5]:
print("Model Type:", config.model_type)
print("Hidden Size:", config.hidden_size)
print("Vocabulary Size:", config.vocab_size)
print("Number Of Layers:", config.num_hidden_layers)

Model Type: llama
Hidden Size: 2048
Vocabulary Size: 32000
Number Of Layers: 22


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_tokens = tokenizer.eos_token
tokenizer.padding_side = 'right'

In [9]:
model = AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.float16,device_map="auto")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
dataset = load_dataset("mlabonne/guanaco-llama2-1k",split="train")

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [11]:
print("Raw Dataset Size:", len(dataset))

Raw Dataset Size: 1000


In [12]:
def clean_dataset(example):

    cleaned_text = example['text'].strip()

    return { "text": cleaned_text}


In [13]:
dataset = dataset.map(clean_dataset)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [14]:
dataset = dataset.train_test_split(test_size=0.1, shuffle=True, seed=42)

In [15]:
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [16]:
print("Train Size:", len(train_dataset))
print("Eval Size:", len(eval_dataset))

Train Size: 900
Eval Size: 100


In [17]:
def tokenize_function(example):

    return tokenizer(
        example['text'],
        truncation = True,
        padding = 'max_length',
        max_length = 128
    )


In [18]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [20]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,

    weight_decay=0.001,

    warmup_ratio=0.03,

    max_grad_norm=0.3,

    fp16=True,

    save_strategy="epoch",

    eval_strategy="epoch",

    logging_steps=25,

    report_to="none"
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_eval
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
prompt = "Explain machine learning."
